In [31]:
import asyncio
from agno.knowledge import Knowledge
from agno.knowledge.embedder.huggingface import HuggingfaceCustomEmbedder
from agno.agent import Agent
from agno.vectordb.lancedb import LanceDb
from agno.vectordb.search import SearchType
from dotenv import load_dotenv

load_dotenv()

True

In [32]:
from agno.models.ollama import Ollama
from agno.knowledge.embedder.sentence_transformer import SentenceTransformerEmbedder

model = Ollama(id="gemma4:31b-cloud")


# Defaults to "sentence-transformers/all-MiniLM-L6-v2" (384 dimensions)
local_embedder = SentenceTransformerEmbedder(
    id="sentence-transformers/all-MiniLM-L6-v2",
    dimensions=384
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2786.16it/s]


In [37]:
from agno.knowledge.reranker.sentence_transformer import SentenceTransformerReranker

knowledge = Knowledge(
    name="test-kb",
    vector_db=LanceDb(
        uri="tmp/lancedb",
        table_name="test_docs",
        search_type=SearchType.hybrid,
        embedder=local_embedder,
        reranker=SentenceTransformerReranker(model="cross-encoder/ms-marco-MiniLM-L-6-v2"),
    ),
    max_results=5,
)

In [38]:
from pprint import pprint

def test_search(query: str, limit: int = 5):
    results = knowledge.search(query, max_results=limit)

    print(f"\nQuery: {query!r}")
    print(f"Got {len(results)} results\n")

    pprint(results)
    return results

In [39]:
knowledge.insert(path="sample_kb.md")

INFO Adding content from path, 26a9e2f6-7cca-5d07-b5a3-f3df16c01b45, None, sample_kb.md, None

INFO Deleted 1 records with content_hash 'a63fd7f03197d94aea5fa68d09f2dc32640b5799b0c9154b7d2db172e3ec8271' from   
     table 'test_docs'.

In [40]:
test_search("Whats the refund window for enterprise customers?")

d:\Cluvo_backend\ksp\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Aaadrish\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2411.88it/s]


INFO Found 1 documents


Query: 'Whats the refund window for enterprise customers?'
Got 1 results

[Document(content='# Acme Corp Support Handbook (Test Document) ## Refund '
                  'Policy Customers may request a refund within 30 days of '
                  'purchase, provided the product is unused and in its '
                  'original packaging. Refunds are processed within 5-7 '
                  'business days to the original payment method. Enterprise '
                  'customers on annual contracts are eligible for prorated '
                  'refunds only if the cancellation request is submitted in '
                  'writing within the first 60 days of the contract term. ## '
                  'Shipping Policy Standard shipping takes 3-5 business days '
                  'within the continental US. Express shipping (1-2 business '
                  'days) is available for an additional fee. International '
                  'shipping times vary by destination and customs processing, 

[Document(content='# Acme Corp Support Handbook (Test Document) ## Refund Policy Customers may request a refund within 30 days of purchase, provided the product is unused and in its original packaging. Refunds are processed within 5-7 business days to the original payment method. Enterprise customers on annual contracts are eligible for prorated refunds only if the cancellation request is submitted in writing within the first 60 days of the contract term. ## Shipping Policy Standard shipping takes 3-5 business days within the continental US. Express shipping (1-2 business days) is available for an additional fee. International shipping times vary by destination and customs processing, typically 7-14 business days. ## Warranty Information All hardware products come with a 1-year limited warranty covering manufacturing defects. The warranty does not cover damage from misuse, accidents, or unauthorized modifications. To file a warranty claim, customers must provide proof of purchase and a

In [26]:
from agno.run.agent import RunOutput

def add_citations(run_output: RunOutput) -> None:
    """
    Deterministic post-hook: appends source citations to the final answer
    based on structured retrieval metadata — never relies on the model
    mentioning sources itself.
    """
    if not run_output.references:
        return  # no knowledge base was searched this run, nothing to cite

    # collect unique (name, chunk) pairs across all search calls made this run
    seen = set()
    sources = []
    for ref_block in run_output.references:
        for ref in ref_block.references:
            name = ref.get("name", "unknown")
            chunk = ref.get("meta_data", {}).get("chunk")
            key = (name, chunk)
            if key not in seen:
                seen.add(key)
                sources.append(f"{name} (chunk {chunk})" if chunk is not None else name)

    if not sources:
        return

    citation_block = "\n\n**Sources:**\n" + "\n".join(f"- {s}" for s in sources)
    run_output.content = (run_output.content or "") + citation_block

In [27]:
agent = Agent(
    model=model,
    knowledge=knowledge,
    search_knowledge=True,
    # Format knowledge references as YAML instead of the default JSON
    references_format="yaml",
    post_hooks=[add_citations],
    # markdown=True,
)

In [28]:
response = agent.run("whats the refund window for enterprise customers?")

INFO Found 1 documents

In [29]:
pprint(response.content)

('Enterprise customers on annual contracts are eligible for prorated refunds '
 'if the cancellation request is submitted in writing within the first 60 days '
 'of the contract term.\n'
 '\n'
 '**Sources:**\n'
 '- sample_kb.md (chunk 1)')
